# Shared session

Two sessions on one port, and who else is attached.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

The first session on a port spawns a broker (`tools/session.py`) and every later one attaches to it on loopback port 8763; the broker owns the port and answers the board's 10 s deadman every 3 s for an attached client.

In [2]:
from coaxial import Coaxial63100, broker

print('serving:', broker.serving())
print('clients before:', broker.clients())

serving: None


clients before: None


In [3]:
first = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
second = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(first)
print(second)
print('clients now:', broker.clients())

<Coaxial63100 Simulated SIMULATED>
<Coaxial63100 Simulated SIMULATED>


clients now: None


`device.link` is the port policy this session was opened with; the board's own link subsystem is `device.board.link`.

In [4]:
print(first.origin.interface, '|', first.system.version()['description'])
print('opened with link=%r on %r' % (first.link, first.port))
print(second.board.link.stats())

simulated | SIMULATED three-phase BLDC inverter at the pelvis
opened with link='auto' on 'COM4'
{'port': 0, 'name': 'USART3', 'rs485': False, 'open': True, 'baud': 115200, 'unit_id': 1, 't15_ticks': 1750, 't35_ticks': 4083, 'bus_message': 42, 'bus_comm_error': 0, 'server_message': 42, 'server_exception': 0, 'server_no_response': 0, 'char_overrun': 0, 'ring_dropped': 0, 'for_others': 0}


Closing one session leaves the other's board alone: the stage is disarmed on the way out only by the session that armed it, or when nobody else is left.

A cooked reading claims a physical quantity, so it refuses with the front end off: mid-scale would put the NTC at exactly 25.00 C and the DC link at a plausible number that is not a measurement (invariant 9). `daq.enable()` takes a reference this session's `close()` releases.

In [5]:
from coaxial.errors import DeviceStateError

try:
    print(first.analog.scan())
except DeviceStateError as exc:
    print('refused:', exc)

daq = first.daq
daq.enable()
print(first.analog.scan())

refused: the scan reports the analog front end off, so every channel read mid-scale: ntc_centidegc would be exactly 2500 and dcbus_mv a plausible number that is not a measurement. Call board.afe.enable() first.
{'phase_u_raw': 8687, 'phase_v_raw': -16221, 'phase_w_raw': 1236, 'dcbus_raw': 20800, 'dcbus_mv': 24803, 'ntc_raw': 36309, 'ntc_centidegc': 3081, 'afe_on': True, 'pe15': False}


In [6]:
second.close()
print('clients after one close:', broker.clients())
first.close()
print('clients after both:', broker.clients())

clients after one close: None


clients after both: None


## Conclusions

In [7]:
for name, session in (('first', first), ('second', second)):
    print('%-7s %-22s simulated=%s' % (name, session.origin.label, session.simulated))
print('broker serving:', broker.serving())
print('port asked for: %r  link policy: %r' % (first.port, first.link))

first   Simulated              simulated=True
second  Simulated              simulated=True
broker serving: None
port asked for: 'COM4'  link policy: 'auto'


On a real port the first session spawns a broker in its own process (`tools/session.py`) and every later one attaches over loopback; the broker owns the port, hands the console over once, and answers the board's 10 s deadman every 3 s for an attached client, so a session thinking between turns keeps its rail claims and its armed stage. Opening through a live broker is 0.05 s against 5.85 s starting one, which is why it lingers 45 s after the last client (FINDINGS).

The stage is the board's, not a session's: `close()` disarms what this session armed, and otherwise only when nobody else is left - three switching runs once ended the moment a second session asked the board an unrelated question.

`broker.clients()` is None when nothing is serving the port: asking is not using, so the count it reports attaches, asks and closes, and does not include itself.